In [0]:
/* ============================================================================
   KPI_QUERIES.SQL
   NovaBC Bank — Customer Churn Analytics

   Purpose:
   Advanced Spark SQL analysis built on top of churn_silver.vw_churn_master.

   The queries below explore churn patterns using:
   - CTEs for step-by-step transformations
   - Window functions for ranking and customer comparisons
   - Subqueries for benchmark analysis
   - UNION ALL for creating Power BI-friendly datasets

   These are analytical queries created separately from the Gold layer tables
   to demonstrate deeper SQL problem-solving skills.
============================================================================ */


/* ============================================================================
   1. Churn Rate by Engagement Segment
   ----------------------------------------------------------------------------
   Business question:
   Which engagement groups are showing the highest customer churn?

   Approach:
   Build the analysis in layers:
   1. Start with the complete customer population
   2. Filter customers who left
   3. Compare churn across engagement groups
============================================================================ */

WITH base_population AS (

    SELECT
        id,
        exit,
        engagement_tier,
        churn_risk_label,
        balance_tier
    FROM churn_silver.vw_churn_master

),

churned_customers AS (

    SELECT *
    FROM base_population
    WHERE exit = true

),

engagement_summary AS (

    SELECT
        b.engagement_tier,
        COUNT(*) AS total_customers,
        COUNT(c.id) AS churned_customers,

        ROUND(
            COUNT(c.id) / COUNT(*) * 100,
            2
        ) AS churn_rate_pct

    FROM base_population b

    LEFT JOIN churned_customers c
        ON b.id = c.id

    GROUP BY b.engagement_tier

)

SELECT *
FROM engagement_summary
ORDER BY churn_rate_pct DESC;



/* ============================================================================
   2. Ranking Provinces by Churn Rate
   ----------------------------------------------------------------------------
   Business question:
   Which provinces are losing the most customers?

   Ranking helps identify regions that need retention attention.
============================================================================ */

WITH province_churn AS (

    SELECT
        origin_province,

        COUNT(*) AS total_customers,

        SUM(
            CASE 
                WHEN exit = true THEN 1 
                ELSE 0 
            END
        ) AS churned_customers,

        ROUND(
            SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        ) AS churn_rate_pct

    FROM churn_silver.vw_churn_master

    GROUP BY origin_province

)

SELECT
    origin_province,
    total_customers,
    churned_customers,
    churn_rate_pct,

    RANK() OVER(
        ORDER BY churn_rate_pct DESC
    ) AS churn_rank,

    DENSE_RANK() OVER(
        ORDER BY churn_rate_pct DESC
    ) AS churn_dense_rank

FROM province_churn

ORDER BY churn_rank;



/* ============================================================================
   3. Customer Risk Quartile Analysis
   ----------------------------------------------------------------------------
   Business question:
   Does customer churn increase as risk score increases?

   Customers are divided into four groups to compare risk levels.
============================================================================ */

WITH risk_groups AS (

    SELECT
        id,
        exit,
        risk_score,

        NTILE(4) OVER(
            ORDER BY risk_score
        ) AS risk_quartile

    FROM churn_silver.vw_churn_master

)

SELECT
    risk_quartile,

    COUNT(*) AS total_customers,

    SUM(
        CASE 
            WHEN exit = true THEN 1 
            ELSE 0 
        END
    ) AS churned_customers,

    ROUND(
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)
        / COUNT(*) * 100,
        2
    ) AS churn_rate_pct,

    ROUND(MIN(risk_score),3) AS min_risk_score,
    ROUND(MAX(risk_score),3) AS max_risk_score

FROM risk_groups

GROUP BY risk_quartile

ORDER BY risk_quartile;



/* ============================================================================
   4. Highest Risk Customers Within Each Occupation
   ----------------------------------------------------------------------------
   Business question:
   Which existing customers have the highest risk inside their peer group?

   Useful for targeted retention campaigns.
============================================================================ */

WITH customer_ranking AS (

    SELECT
        id,
        occupation,
        risk_score,
        engagement_score,
        churn_risk_label,
        exit,

        RANK() OVER(
            PARTITION BY occupation
            ORDER BY risk_score DESC
        ) AS risk_rank

    FROM churn_silver.vw_churn_master

    WHERE exit = false

)

SELECT *

FROM customer_ranking

WHERE risk_rank <= 3

ORDER BY
    occupation,
    risk_rank;



/* ============================================================================
   5. Running Contribution of Churn by Province
   ----------------------------------------------------------------------------
   Business question:
   Which provinces contribute most of the total churn?

   Helps prioritize regions using an 80/20 style approach.
============================================================================ */

WITH province_churn AS (

    SELECT
        origin_province,

        SUM(
            CASE 
                WHEN exit = true THEN 1
                ELSE 0
            END
        ) AS churned_customers,

        ROUND(
            SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)
            /
            (
                SELECT SUM(
                    CASE WHEN exit = true THEN 1 ELSE 0 END
                )
                FROM churn_silver.vw_churn_master
            )
            * 100,
            2
        ) AS pct_of_total_churn

    FROM churn_silver.vw_churn_master

    GROUP BY origin_province

)

SELECT

    origin_province,
    churned_customers,
    pct_of_total_churn,

    SUM(churned_customers) OVER(
        ORDER BY churned_customers DESC
        ROWS BETWEEN UNBOUNDED PRECEDING 
        AND CURRENT ROW
    ) AS running_churn_total,


    ROUND(
        SUM(pct_of_total_churn) OVER(
            ORDER BY churned_customers DESC
            ROWS BETWEEN UNBOUNDED PRECEDING 
            AND CURRENT ROW
        ),
        2
    ) AS running_churn_percentage


FROM province_churn

ORDER BY churned_customers DESC;



/* ============================================================================
   6. Customers Above Average Risk
   ----------------------------------------------------------------------------
   Business question:
   Which customers have risk scores higher than the bank average?

   Used to identify customers requiring closer monitoring.
============================================================================ */

SELECT

    id,
    occupation,
    origin_province,
    risk_score,
    engagement_score,
    churn_risk_label,
    exit

FROM churn_silver.vw_churn_master

WHERE risk_score >
(
    SELECT AVG(risk_score)
    FROM churn_silver.vw_churn_master
)

ORDER BY risk_score DESC;



/* ============================================================================
   7. Churned Customers Above Their Occupation Balance Average
   ----------------------------------------------------------------------------
   Business question:
   Are we losing valuable customers who hold higher balances
   compared with others in their occupation?
============================================================================ */

SELECT

    m.id,
    m.occupation,
    m.balance,
    m.churn_reason_inferred

FROM churn_silver.vw_churn_master m

WHERE m.exit = true

AND m.balance >
(
    SELECT AVG(m2.balance)

    FROM churn_silver.vw_churn_master m2

    WHERE m2.occupation = m.occupation
)

ORDER BY
    m.occupation,
    m.balance DESC;



/* ============================================================================
   8. Provinces With Above Average High-Risk Churn
   ----------------------------------------------------------------------------
   Business question:
   Which regions have a bigger high-risk churn problem than the bank average?
============================================================================ */

WITH province_high_risk AS (

    SELECT

        origin_province,

        COUNT(*) AS high_risk_customers,

        SUM(
            CASE WHEN exit = true THEN 1 ELSE 0 END
        ) AS high_risk_churned,

        ROUND(
            SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        ) AS high_risk_churn_rate


    FROM churn_silver.vw_churn_master

    WHERE churn_risk_label = 'High Risk'

    GROUP BY origin_province

),

overall_high_risk AS (

    SELECT

        ROUND(
            SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        ) AS bank_high_risk_rate

    FROM churn_silver.vw_churn_master

    WHERE churn_risk_label = 'High Risk'

)

SELECT

    p.origin_province,
    p.high_risk_customers,
    p.high_risk_churned,
    p.high_risk_churn_rate,
    o.bank_high_risk_rate


FROM province_high_risk p

CROSS JOIN overall_high_risk o

WHERE p.high_risk_churn_rate > o.bank_high_risk_rate

ORDER BY p.high_risk_churn_rate DESC;



/* ============================================================================
   9. Unified Segment Analysis
   ----------------------------------------------------------------------------
   Business question:
   Compare churn across different customer dimensions in one dataset.

   This output can directly feed Power BI visuals.
============================================================================ */

WITH segment_data AS (

    SELECT
        'Age Group' AS segment_type,
        age_group AS segment_value,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN exit=true THEN 1 ELSE 0 END) AS churned,
        ROUND(
            SUM(CASE WHEN exit=true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        ) AS churn_rate_pct

    FROM churn_silver.vw_churn_master

    GROUP BY age_group


    UNION ALL


    SELECT
        'Balance Tier',
        balance_tier,
        COUNT(*),
        SUM(CASE WHEN exit=true THEN 1 ELSE 0 END),
        ROUND(
            SUM(CASE WHEN exit=true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        )

    FROM churn_silver.vw_churn_master

    GROUP BY balance_tier


    UNION ALL


    SELECT
        'Engagement Tier',
        engagement_tier,
        COUNT(*),
        SUM(CASE WHEN exit=true THEN 1 ELSE 0 END),
        ROUND(
            SUM(CASE WHEN exit=true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        )

    FROM churn_silver.vw_churn_master

    GROUP BY engagement_tier

)

SELECT *

FROM segment_data

ORDER BY churn_rate_pct DESC;



/* ============================================================================
   10. Churn Difference Between Risk Levels
   ----------------------------------------------------------------------------
   Business question:
   How much does churn increase between customer risk categories?

   Helps quantify the impact of risk segmentation.
============================================================================ */

WITH risk_summary AS (

    SELECT

        churn_risk_label,

        ROUND(
            SUM(CASE WHEN exit=true THEN 1 ELSE 0 END)
            / COUNT(*) * 100,
            2
        ) AS churn_rate_pct


    FROM churn_silver.vw_churn_master

    GROUP BY churn_risk_label

)

SELECT

    churn_risk_label,

    churn_rate_pct,

    LAG(churn_rate_pct) OVER(
        ORDER BY churn_rate_pct
    ) AS previous_risk_rate,


    ROUND(
        churn_rate_pct -
        LAG(churn_rate_pct) OVER(
            ORDER BY churn_rate_pct
        ),
        2
    ) AS churn_increase_points,


    ROUND(
        churn_rate_pct /
        LAG(churn_rate_pct) OVER(
            ORDER BY churn_rate_pct
        ),
        2
    ) AS increase_multiplier


FROM risk_summary

ORDER BY churn_rate_pct;